# Notebook 2: Features and Labels

This notebook turns the clean daily prices from Notebook 1 into the monthly table that the models are trained on. For every ETF at every month-end it computes seven technical indicators (the model inputs) and two labels (what the models learn to predict). It then checks the table for errors, sets up the training windows for the walk-forward test, and describes the 2010 to 2019 data.

**Work packages:** WBS 2.4 (Features and labels), 2.5 (Variables and exploration) and 3.1 (Training windows)

**Input:** `Daily Prices.parquet` from Notebook 1

**Outputs** (Google Drive, `Capstone/data`):

| File | Folder | Contents |
|---|---|---|
| Monthly Data.parquet / .csv | processed | One row per universe, ETF and month-end: 7 features and 2 labels |
| Training Windows.parquet / .csv | processed | Training and validation months for each of the 81 test months |
| Variables.csv | processed | Definition of every column |
| Monthly Data Checks.csv | reports | Results of all checks on the monthly table |
| Exploration tables | reports | Summary statistics for 2010 to 2019 |

**The two labels**

| Label | Definition | Used by |
|---|---|---|
| Next-month return | Return from this month-end to the next | Model A (regression) and the portfolio accounting |
| Top 3 | 1 if the ETF finishes among the 3 best of its universe next month, otherwise 0 | Model B (classification) |

**Timing rule:** features use only prices up to the month-end; labels describe the following month. A row can therefore be used for training only after its following month has ended. Section 4 tests this directly.

**How to run:** run Notebook 1 first, then Runtime > Run all. A full run takes about one minute.

## 1. Setup

Connects Google Drive, where all data files are stored, and loads the project code from the GitHub repository. Run this cell first in every new Colab session. When Colab asks for access, choose the Google account that holds the `Capstone/data` folder.

In [ ]:
# Connect Google Drive. All data files are read from and saved to
# MyDrive/Capstone/data.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# Download the project code on the first run of a session; on later runs,
# update it so any changes pushed to GitHub are picked up.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

# Install the pinned library versions listed in requirements.txt.
%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make the project modules importable with "from src import ...".
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, download, cleaning, features, windows, variables

# Reload the modules so code changes pulled from GitHub take effect
# without restarting the Colab runtime.
for module in (config, download, cleaning, features, windows, variables):
    importlib.reload(module)

DIRS = config.data_dirs()
pd.set_option("display.width", 160)

## 2. Load the daily prices

Reads the clean daily prices saved by Notebook 1 and confirms that the table still matches its definition in the Variables list (same columns, types and value ranges).

In [ ]:
clean = cleaning.load_clean(DIRS["processed"])

# The checksum of the frozen download is stored with the monthly table, so
# every result can be traced back to the exact raw file it came from.
raw_info = json.loads((DIRS["raw"] / config.FILES["raw_info"]).read_text())

print(f"{len(clean):,} rows, {clean['ticker'].nunique()} ETFs, "
      f"{clean['date'].min():%Y-%m-%d} to {clean['date'].max():%Y-%m-%d}")
print("Raw file checksum:", raw_info["md5"])
cleaning.summarize(variables.check_table(clean, "Daily Prices").rename(columns={"column": "ticker"}))

## 3. Features and monthly table

Computes the seven indicators from the paper for every ETF on every trading day, using adjusted prices so that dividends do not appear as price drops. The indicators are then taken at each month-end and combined with the two labels.

| Feature | Measures | Window |
|---|---|---|
| Momentum | Return over the last month | 21 days |
| Trend | 10-day average relative to 50-day average | 50 days |
| Price bias | Price relative to its 10-day average | 10 days |
| RSI | Balance of recent gains and losses (0 to 100) | 14 days |
| MACD | Whether short-term momentum is speeding up or slowing, relative to price | 35 days |
| Bollinger width | Recent volatility relative to price | 20 days |
| Volume bias | Trading volume relative to its 10-day average | 10 days |

The table holds both universes. SPY and XLK appear in each with the same features but separate labels, because the top 3 is decided within each universe. The latest month (September 2026) has features but no labels yet.

In [ ]:
# Seven indicators for every ETF and trading day.
daily = features.daily_features(clean)

# Month-end rows for both universes, with labels attached.
panel = features.build_panel(clean, daily)

# Rows per universe and period: history (2010 to 2019, training only),
# test (2020 to 2026) and latest (no label yet).
print(panel.groupby(["universe", "period"]).size().unstack(fill_value=0))
print("Month-ends:", panel["date"].nunique())
print("Rows where volume bias was set to 0 (no trading that day):", int(panel["volume_filled"].sum()))
panel.head()

## 4. Checks on the monthly table

Confirms the table is complete and correct: expected row counts, no duplicates or missing values, valid value ranges, exactly three top-3 labels per month, and next-month returns that match an independent recalculation.

The **look-ahead test** recomputes the features at randomly chosen month-ends using only prices available up to that date. Any difference from the stored values would mean a feature had used future prices. Every column is also checked against the Variables list. The notebook stops on any FAIL.

In [ ]:
panel_checks = features.validate_panel(panel, clean)
column_checks = variables.check_table(panel, "Monthly Data")

print(panel_checks.to_string(index=False))
print()
print(column_checks.to_string(index=False))

all_checks = pd.concat([panel_checks.assign(column=""), column_checks], ignore_index=True)
if (all_checks["status"] == "FAIL").any():
    raise RuntimeError("A check failed. The monthly table should not be used.")

## 5. Training windows

Sets out, for each of the 81 test months (January 2020 to September 2026), which earlier months the models are trained on. A month can be used for training only if its label is already known on the decision date. The last 12 months of each window are used to tune model settings.

Three window types are saved: **growing** (all months since 2010, used in the main test) and **36** and **60** months (rolling windows, used as robustness checks).

In [ ]:
all_windows = {w: windows.training_windows(panel, w) for w in ["growing", *config.ROLLING_WINDOWS]}
window_table = pd.concat(all_windows.values(), ignore_index=True)

# Each window type must cover all 81 test months and always end training
# before the decision date.
for w, table in all_windows.items():
    print(f"Window: {w}")
    print(windows.check_windows(table, panel).to_string(index=False))
    print()
print("Column checks:", variables.check_table(window_table, "Training Windows")["status"].value_counts().to_dict())

# First two and last test months of the main (growing) design.
all_windows["growing"].iloc[[0, 1, -1]]

## 6. Save

Saves the monthly table, the training windows and the Variables list. An information file records the checksum of the raw download and the main settings used to build the table.

In [ ]:
out = DIRS["processed"]
panel.to_parquet(out / config.FILES["monthly"], index=False)
panel.to_csv(out / "Monthly Data.csv", index=False)
window_table.to_parquet(out / config.FILES["windows"], index=False)
window_table.to_csv(out / "Training Windows.csv", index=False)
variables.to_frame().to_csv(out / config.FILES["variables"], index=False)
all_checks.to_csv(DIRS["reports"] / config.FILES["checks"], index=False)

info = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "raw_file_md5": raw_info["md5"],
    "rows": {u: int(n) for u, n in panel["universe"].value_counts().items()},
    "month_ends": int(panel["date"].nunique()),
    "features": config.FEATURES,
    "top_k": config.TOP_K,
    "test_months": [config.TEST_FIRST_MONTH, config.TEST_LAST_MONTH],
}
(out / config.FILES["monthly_info"]).write_text(json.dumps(info, indent=2))
print("Saved: Monthly Data, Training Windows, Variables, Monthly Data Checks")

## 7. Exploration of 2010 to 2019 (ETF_A)

Describes the data before any model is built. Only the 2010 to 2019 history is used, so the test period stays unseen until the main test is fixed (WBS 4.3). These results describe the data and are not used to select features.

### 7a. Feature summary

Distribution of each feature. Values should be small for the ratio features and between 0 and 100 for RSI.

In [ ]:
hist = panel[(panel["period"] == "history") & (panel["universe"] == "ETF_A")].copy()
print(f"{hist['date'].nunique()} months, {len(hist)} rows")

summary = hist[config.FEATURES].describe(percentiles=[0.05, 0.5, 0.95]).T
summary.to_csv(DIRS["reports"] / "Exploration Feature Summary.csv")
summary.round(4)

### 7b. Top-3 frequency and persistence

How often each ETF finishes in the top 3. By chance alone each ETF would do so in 3 of 8 months (37.5%). Persistence shows how often an ETF that is in the top 3 this month is in the top 3 again next month: a value well above 37.5% would mean rankings carry over from month to month.

In [ ]:
top3_share = hist.groupby("ticker")["top3"].mean().sort_values(ascending=False)

# Whether each ETF was in the top 3 in the month just ended, taken from the
# previous row's label.
hist = hist.sort_values(["ticker", "date"])
hist["top3_now"] = hist.groupby("ticker")["top3"].shift(1)
stay = hist.dropna(subset=["top3_now"]).groupby("top3_now")["top3"].mean()

print(top3_share.round(3).to_string())
print()
print(f"In the top 3 next month, if in the top 3 now:     {stay.get(1.0, np.nan):.3f}")
print(f"In the top 3 next month, if not in the top 3 now: {stay.get(0.0, np.nan):.3f}")
top3_share.rename("share of months in top 3").to_csv(DIRS["reports"] / "Exploration Top 3 Share.csv")

### 7c. How well each feature orders next month's returns

For every month, the rank correlation between each feature and next month's return across the eight ETFs, averaged over all months. Positive values mean ETFs with a higher feature value tended to earn more the following month. With only eight ETFs per month, single months are noisy; the average and the t-statistic are what matter. Values close to zero are normal for monthly returns.

In [ ]:
def monthly_rank_correlation(df, feature):
    """Spearman correlation between a feature and next month's return, one value per month."""
    return df.groupby("date").apply(
        lambda g: g[feature].corr(g["next_return"], method="spearman"),
        include_groups=False)

ic = pd.DataFrame({f: monthly_rank_correlation(hist, f) for f in config.FEATURES})
ic_table = pd.DataFrame({
    "mean correlation": ic.mean(),
    "std": ic.std(),
    "t-stat": ic.mean() / (ic.std() / np.sqrt(len(ic))),
    "share of months > 0": (ic > 0).mean(),
}).round(3)
ic_table.to_csv(DIRS["reports"] / "Exploration Feature Correlation.csv")
ic_table

### 7d. Correlation between features

Features built from the same prices tend to move together. High correlations matter for the linear models, which can give unstable weights to overlapping inputs; tree models are less affected.

In [ ]:
corr = hist[config.FEATURES].corr(method="spearman")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation between features (ETF_A, 2010 to 2019)")
plt.tight_layout()
plt.show()

### 7e. Value of a correct ranking

The average return of the true top 3 minus the average return of the other five ETFs, each month. This is the most a perfect ranking could gain before fees. Replacing one of the three holdings costs 0.33% in fees, so a model needs a real edge to beat simply holding its positions.

In [ ]:
gap = hist.groupby(["date", "top3"])["next_return"].mean().unstack()
gap = gap[1] - gap[0]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(gap.index, gap * 100, width=20, color="#4a6572")
ax.axhline(gap.mean() * 100, color="black", linewidth=1, linestyle="--",
           label=f"average {gap.mean():.2%} per month")
ax.set_ylabel("top 3 minus other 5 (%)")
ax.set_title("Monthly return gap between the true top 3 and the other 5 ETFs (2010 to 2019)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary

Key figures for the process log and the Data Specification sheet. After the run, save the notebook (File > Save) so the outputs are kept.

In [ ]:
passed = int((all_checks["status"] == "PASS").sum())
failed = int((all_checks["status"] == "FAIL").sum())
print(f"Month-ends:   {panel['date'].nunique()} ({panel['date'].min():%Y-%m} to {panel['date'].max():%Y-%m})")
for u, n in panel["universe"].value_counts().sort_index().items():
    print(f"Rows {u}:   {n:,}")
print(f"Test months:  {len(all_windows['growing'])} ({config.TEST_FIRST_MONTH} to {config.TEST_LAST_MONTH})")
print(f"Checks:       {passed} passed, {failed} failed")